# Tools and Agents

In [12]:
from langchain_experimental.utilities import PythonREPL

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.tools import Tool, tool
from langchain_core.prompts import PromptTemplate

from langchain_openai import ChatOpenAI
from IPython.display import display, HTML, Markdown

In [2]:
python_repl = PythonREPL() # This provides an environment where Python code can be executed as strings. Pass string of Python code and execute it

# Create a Tool using the Tool class
python_calculator = Tool(
    name="Python Calculator", # The name of the tool - this helps agents identify when to use this tool
    func=python_repl.run, # Callback function when tool is used
    description="Useful for when you need to perform calculations or execute Python code. Input should be valid Python code."
    # A description of what the tool does and how to use it - This helps the agent understand when and how to use this tool
)

In [3]:
# Let's test this tool with a simple Python command:
python_calculator.invoke("a = 3; b = 1; print(a+b)")

Python REPL can execute arbitrary code. Use with caution.


'4\n'

In [5]:
# Create a Tool using the @tool decorator

@tool
def search_weather(location: str):
    """Search for the current weather in the specified location."""
    return f"The weather in {location} is currently sunny and 72°F." # In a real application, this would call a weather API

## Toolkits

Toolkits are collections of tools that are designed to be used together for specific tasks.

In [13]:
# Create a toolkit (collection of tools)
tools = [python_calculator, search_weather]

In [14]:
tools

[Tool(name='Python Calculator', description='Useful for when you need to perform calculations or execute Python code. Input should be valid Python code.', func=<bound method PythonREPL.run of PythonREPL(globals={'__builtins__': {'__name__': 'builtins', '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.", '__package__': '', '__loader__': <class '_frozen_importlib.BuiltinImporter'>, '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'), '__build_class__': <built-in function __build_class__>, '__import__': <built-in function __import__>, 'abs'

## Agents

In [ ]:
# Model configuration
openai_llm = ChatOpenAI(model="gpt-4.1-nano")

In [9]:
# Create the ReAct agent prompt template
# The ReAct prompt needs to instruct the model to follow the thought-action-observation pattern
prompt_template = """You are an agent who has access to the following tools:

{tools}

The available tools are: {tool_names}

To use a tool, please use the following format:
```
Thought: I need to figure out what to do
Action: tool_name
Action Input: the input to the tool
```

After you use a tool, the observation will be provided to you:
```
Observation: result of the tool
```

Then you should continue with the thought-action-observation cycle until you have enough information to respond to the user's request directly.
When you have the final answer, respond in this format:
```
Thought: I know the answer
Final Answer: the final answer to the original query
```

Remember, when using the Python Calculator tool, the input must be valid Python code.

Begin!

Question: {input}
{agent_scratchpad}
"""

prompt = PromptTemplate.from_template(prompt_template)

The **create_react_agent** function creates an agent that follows the Reasoning + Acting (ReAct) framework

**How ReAct Works**:
The ReAct framework follows a specific cycle:

- Reasoning: The agent thinks about the problem and plans its approach
- Action: It selects a tool and formulates the input
- Observation: It receives the result of the tool execution
- Repeat: It reasons about the observation and decides the next step


**Output Format Control**:
The ReAct agent must produce output in a structured format that includes:

- Thought: The agent's reasoning process
- Action: The tool to use
- Action Input: The input to the tool
- Observation: The result of the tool execution
- Final Answer: The final response when the agent has solved the problem

In [15]:
# Create the agent
agent = create_react_agent(
    llm=openai_llm,
    tools=tools,
    prompt=prompt
)

In [ ]:
# Create the agent executor
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, # displays the entire thought process
    handle_parsing_errors=True # Can implement retry logic for failed tool executions
)

In [ ]:
# Ask the agent a question that requires only calculation
result = agent_executor.invoke({"input": "What is the square root of 256?"})
print(result["output"])



> Entering new AgentExecutor chain...
Thought: I need to calculate the square root of 256. I'll use Python to perform this calculation.
Action: Python Calculator
Action Input: import math; math.sqrt(256)Thought: I will now interpret the result of the calculation to provide the answer.
Final Answer: The square root of 256 is 16.

> Finished chain.
The square root of 256 is 16.


In [18]:
# Example of; Multiple tools calling in same prompt
result = agent_executor.invoke({"input": "What is the weather in Pakistan? Convert Weather value from fahrenheit to celcius."})
print(result["output"])



> Entering new AgentExecutor chain...
Thought: First, I need to find the current weather in Pakistan to get the temperature in Fahrenheit. Then, I will convert that temperature to Celsius.  
Action: search_weather  
Action Input: PakistanThe weather in Pakistan is currently sunny and 72°F.Thought: The current temperature in Pakistan is 72°F. To convert this to Celsius, I will use the formula: (Fahrenheit - 32) * 5/9. I will perform this calculation using the Python Calculator tool.  
Action: Python Calculator  
Action Input: (72 - 32) * 5/9Thought: The calculation has been performed to convert 72°F to Celsius. The result will tell me the temperature in Celsius.  
Final Answer: The current weather in Pakistan is sunny, and the temperature is approximately 22.22°C.

> Finished chain.
The current weather in Pakistan is sunny, and the temperature is approximately 22.22°C.


## Exercise - Creating Your First LangChain Agent with Basic Tools

In [ ]:
# Create a simple calculator tool
def calculator(expression: str) -> str:
    """A simple calculator that can add, subtract, multiply, or divide two numbers.
    Input should be a mathematical expression like '2 + 2' or '15 / 3'."""
    try:
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error calculating: {str(e)}"

# Create a text formatting tool
def format_text(text: str) -> str:
    """Format text to uppercase, lowercase, or title case.
    Input should be in format: '[format_type]: [text]'
    where format_type is 'uppercase', 'lowercase', or 'titlecase'.
    
    Examples:
    - uppercase: hello world -> HELLO WORLD
    - lowercase: HELLO WORLD -> hello world 
    - titlecase: hello world -> Hello World
    """
    try:
        if ":" in text:
            format_type, content = text.split(":", 1) # Split only on the first colon
            format_type = format_type.strip().lower()
            content = content.strip()
        else:
            # If no colon, assume they want titlecase
            return f"Missing format. Example: titlecase: {text} -> {text.title()}"
            
        if format_type == "uppercase":
            return content.upper()
        elif format_type == "lowercase":
            return content.lower()
        elif format_type == "titlecase":
            return content.title()
        else:
            return f"Unknown format {format_type}. Use: uppercase, lowercase, or titlecase"
        pass
    except Exception as e:
        return f"Error formatting text: {str(e)}"

# Create Tool objects for our functions
tools = [
    Tool(
        name="calculator",
        func=calculator,
        description="Useful for performing simple math calculations"
    ),
    Tool(
        name="format_text",
        func=format_text,
        description="Useful for formatting text to uppercase, lowercase, or titlecase"
    )
]

# Create a simple prompt template
prompt_template = """You are a helpful assistant who can use tools to help with simple tasks.
You have access to these tools:

{tools}

The available tools are: {tool_names}

Follow this format:

Question: the user's question
Thought: think about what to do
Action: the tool to use, should be one of [{tool_names}]
Action Input: the input to the tool
Observation: the result from the tool
Thought: I now know the final answer
Final Answer: your final answer to the user's question

Question: {input}
{agent_scratchpad}
"""

# Create the agent and executor
prompt = PromptTemplate.from_template(prompt_template)
agent = create_react_agent(
    llm=openai_llm,
    tools=tools,
    prompt=prompt
)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

# Test with simple questions
test_questions = [
    "What is 25 + 63?", 
    "Can you convert 'hello world' to uppercase?",
    "Calculate 15 * 7", 
    "titlecase: langchain is awesome",
]

# Run the tests
for question in test_questions:
    print(f"\n===== Testing: {question} =====")
    result = agent_executor.invoke({"input": question})
    print(f"Final Answer: {result['output']}")


===== Testing: What is 25 + 63? =====


> Entering new AgentExecutor chain...
Thought: I need to perform the addition of 25 and 63.
Action: calculator
Action Input: 25 + 63Result: 88Thought: I know the answer
Final Answer: 88

> Finished chain.
Final Answer: 88

===== Testing: Can you convert 'hello world' to uppercase? =====


> Entering new AgentExecutor chain...
Thought: I need to convert the text 'hello world' to uppercase.
Action: format_text
Action Input: hello worldMissing format. Example: titlecase: hello world -> Hello WorldThought: I need to specify the format as 'uppercase' in the format_text tool.
Action: format_text
Action Input: uppercase: hello worldHELLO WORLDThought: I have successfully converted 'hello world' to uppercase as 'HELLO WORLD'.  
Final Answer: HELLO WORLD

> Finished chain.
Final Answer: HELLO WORLD

===== Testing: Calculate 15 * 7 =====


> Entering new AgentExecutor chain...
Thought: I need to calculate 15 multiplied by 7.
Action: calculator
Action Inpu